# Python descriptors: a second advanced tutorial
### Guided problems, tiny experiments, and fully worked solutions

**A companion to the supplied “Getters and Setters” lesson, not a reprint of the earlier exercise notebook.** The source lesson begins with class-versus-instance access to `__get__`, establishes that descriptor objects are shared by a class's instances, demonstrates the resulting broken `Countdown` and `IntegerValue` implementations, and introduces `__set__`. This notebook starts with that same reasoning approach and goes substantially further.

Every problem is presented as a **tutorial**, not a one-shot code dump. Expect frequent prose-only cells: questions to think through, hints, observations, explanation of state ownership, and edge-case discussions. The **solutions are included**, and every exercise has executable checks.

**How to use it.** Run cells in order from a fresh kernel. For each “Predict” cell, decide what it will do before running it. Some examples intentionally show a *broken design*; its bad behavior is demonstrated with assertions rather than leaving the notebook in an error state. All examples use the Python standard library. Recommended: Python 3.10+.

**Important convention:** a descriptor is placed on the **class**, so it is shared. Per-object data belongs on the target instance (or in a carefully designed external instance-keyed store), not in a single mutable `descriptor.value` field.

## Road map

| Part | Guided problems | Main idea |
|---|---|---|
| I. Understand the protocol | 1–4 | Owner argument, class access, shared state, validation |
| II. Master lookup behavior | 5–8 | Data/non-data precedence, setters, deleters, missing values |
| III. Choose safe storage | 9–11 | Slots, weak references, inheritance and descriptor reuse |
| IV. Design production-grade descriptors | 12–16 | Cache invalidation, atomic updates, coercion, schema fields, concurrency |

At the end you will find extra prediction questions, a debugging checklist, and a final integrated example.

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from threading import Barrier, Lock, Thread
from weakref import WeakKeyDictionary, ref
import gc
import math

_MISSING = object()  # One private sentinel, not a valid user-supplied value.
print("Ready: standard-library-only tutorial")

Ready: standard-library-only tutorial


---
## Problem 01 — A getter knows *who* asked

**Learning objective.** Trace `instance` and `owner`, including inherited descriptors.

**Challenge.** Write an auditing getter that returns itself on class access and reports both arguments on instance access.

Our method is deliberate: make a prediction, run a small experiment, explain what happened, and only then build the final solution.

### Step 1 — Form a prediction

When `Sub.value` looks up a descriptor *inherited* from `Base`, is its `owner` argument `Base` or `Sub`? And what is `instance` when the access starts at a class? Do not rely on the printed memory addresses in the source notebook: inspect **object identity** instead.

In [2]:
class P01Trace:
    def __init__(self):
        self.events = []

    def __get__(self, instance, owner):
        self.events.append((instance, owner))
        return self if instance is None else (instance, owner)


class P01Base:
    value = P01Trace()


class P01Child(P01Base):
    pass

p01_obj = P01Child()
p01_descriptor = P01Base.__dict__["value"]
print(P01Child.value is p01_descriptor)
print(p01_obj.value[0] is p01_obj)
print(p01_obj.value[1] is P01Child)

True
True
True


**Checkpoint / interpretation.** The inherited getter receives the actual accessing class (`P01Child`) as `owner` in this ordinary lookup, even though the descriptor resides in `P01Base.__dict__`. Class access sends `instance=None`. The descriptor is one shared object; the event list here deliberately records diagnostic events, not per-instance field data.

### Step 2 — Solve the audit requirement

Do not use `owner.__name__` as a substitute for class identity: two classes can have the same name. Retain the references and test them with `is`. What should a class-level access return? Returning the descriptor itself makes introspection convenient.

In [3]:
assert P01Child.value is p01_descriptor
assert p01_descriptor.events[0] == (None, P01Child)
assert p01_descriptor.events[1][0] is p01_obj
assert p01_descriptor.events[1][1] is P01Child

another = P01Base()
value, owner = another.value
assert value is another and owner is P01Base
print("Problem 1 solved: all protocol arguments match predictions.")

Problem 1 solved: all protocol arguments match predictions.


### Why this matters

A descriptor’s **defining class** and its **accessing class** are not necessarily the same. Avoid storing the most recently observed `owner` or `instance` on a shared descriptor to represent per-object values: another access would overwrite it. Class access returning `self` is a useful convention, not a protocol requirement.

---
## Problem 02 — A UTC time getter you can actually test

**Learning objective.** Separate dynamic data from descriptor configuration and use aware UTC timestamps.

**Challenge.** Implement a read-only clock field that is evaluated on instance access and can be tested without sleeping.

Our method is deliberate: make a prediction, run a small experiment, explain what happened, and only then build the final solution.

### Step 1 — What is wrong with a fixed expected timestamp?

The input lesson uses `datetime.utcnow().isoformat()`. That yields a naive datetime; modern code should normally prefer `datetime.now(timezone.utc)` for an aware UTC instant. A live clock is also difficult to assert against exactly. Dependency injection solves the testability problem.

In [4]:
class P02UtcTime:
    def __init__(self, clock):
        self._clock = clock  # Configuration shared safely by all instances.

    def __get__(self, instance, owner):
        if instance is None:
            return self
        instant = self._clock()
        if instant.tzinfo is None or instant.utcoffset() is None:
            raise ValueError("Clock must return an aware datetime")
        return instant.astimezone(timezone.utc).isoformat()


fixed = datetime(2030, 1, 2, 3, 4, 5, tzinfo=timezone.utc)
class P02Log:
    recorded_at = P02UtcTime(lambda: fixed)

print(P02Log().recorded_at)

2030-01-02T03:04:05+00:00


### Step 2 — Determine whether the descriptor stores each log's time

It does **not**. Like the first example in the input lesson, the value is computed when read. The injected clock is shared *configuration*; it is not an accidentally shared value. Test class access and two independent log instances.

In [5]:
first, second = P02Log(), P02Log()
assert P02Log.recorded_at is P02Log.__dict__["recorded_at"]
assert first.recorded_at == "2030-01-02T03:04:05+00:00"
assert second.recorded_at == first.recorded_at

class P02BadLog:
    recorded_at = P02UtcTime(lambda: datetime(2030, 1, 1))

try:
    P02BadLog().recorded_at
except ValueError as exc:
    print("Expected rejection:", exc)
else:
    raise AssertionError("A naive clock must be rejected")

Expected rejection: Clock must return an aware datetime


**Checkpoint / interpretation.** The value happens to be equal for two objects because the injected clock is fixed; that does not imply the timestamp was stored on the descriptor. If the requirement were a *creation* timestamp, it should be stored once per log in an initializer or a write-once field instead.

---
## Problem 03 — Find and repair a cross-instance countdown bug

**Learning objective.** Understand why storing mutable instance data on a descriptor leaks values between objects.

**Challenge.** First reproduce the failing rocket countdown, then implement independent counters with clear missing/default semantics.

Our method is deliberate: make a prediction, run a small experiment, explain what happened, and only then build the final solution.

### Step 1 — Diagnose the original mistake

A `Countdown` object attached to `Rocket.countdown` exists once, not once per rocket. If `__get__` updates `self.remaining`, all rockets touch the same integer. What sequence should this intentionally broken implementation produce for rocket A, rocket B, then A again?

In [6]:
class P03BrokenCountdown:
    def __init__(self, start):
        self.remaining = start + 1

    def __get__(self, instance, owner):
        if instance is None:
            return self
        self.remaining -= 1
        return self.remaining

class P03BrokenRocket:
    countdown = P03BrokenCountdown(3)

a, b = P03BrokenRocket(), P03BrokenRocket()
observed = (a.countdown, b.countdown, a.countdown)
print("Shared (incorrect) result:", observed)
assert observed == (3, 2, 1)

Shared (incorrect) result: (3, 2, 1)


### Step 2 — Choose where state belongs

Store the counter in each rocket’s `__dict__`. `__set_name__(owner, name)` learns the field name once when Python creates the containing class. A private storage name avoids colliding with the public descriptor name. For this exercise, use `_countdown_remaining` as the storage key.

In [7]:
class P03Countdown:
    def __init__(self, start):
        if type(start) is not int or start < 0:
            raise ValueError("start must be a nonnegative integer")
        self.start = start  # Shared configuration: safe.

    def __set_name__(self, owner, name):
        self.storage_name = f"_{name}_remaining"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        remaining = instance.__dict__.get(self.storage_name, self.start + 1)
        remaining -= 1
        instance.__dict__[self.storage_name] = remaining
        return remaining

class P03Rocket:
    countdown = P03Countdown(3)

r1, r2 = P03Rocket(), P03Rocket()
print("Independent (correct) result:", r1.countdown, r2.countdown, r1.countdown)

Independent (correct) result: 3 3 2


### Step 3 — Work through what happens on the first read

When `r1.countdown` is evaluated, no private key exists, so the code begins from `start + 1`, subtracts one, and stores `3` **on `r1`**. Reading `r2.countdown` performs the same first-read initialization **on `r2`**. That is the exact distinction missing from the original broken `Countdown`.

In [8]:
x, y = P03Rocket(), P03Rocket()
assert (x.countdown, y.countdown, x.countdown, y.countdown) == (3, 3, 2, 2)
assert x.__dict__["_countdown_remaining"] == 2
assert y.__dict__["_countdown_remaining"] == 2
assert P03Rocket.countdown.start == 3
print("Independent-instance tests passed")

Independent-instance tests passed


### Discussion — What if a client sets `rocket.countdown = 99`?

This version supplies `__get__` but no `__set__`: it is a **non-data descriptor**. A client can put `countdown` in the rocket’s instance dictionary and shadow the descriptor completely. This may be useful for a deliberate override but dangerous for a countdown API. We will learn why and add explicit setter/deleter semantics later.

---
## Problem 04 — Strict, reusable integer coordinates

**Learning objective.** Implement `__set__` and `__get__` with per-instance storage and deliberate type rules.

**Challenge.** Build `Point.x` and `Point.y` fields that reject booleans, floats, and strings rather than silently truncating them.

Our method is deliberate: make a prediction, run a small experiment, explain what happened, and only then build the final solution.

### Step 1 — Decide validation before coding

The source lesson’s `int(value)` quietly converts `1.9` to `1`; `bool` also subclasses `int`, so `isinstance(True, int)` is true. Here, the specification is **exact built-in `int` only**. The getter should raise `AttributeError` if a coordinate has never been assigned. We do not want a fabricated zero to conceal missing data.

In [9]:
print("Boolean passes isinstance(True, int):", isinstance(True, int))
print("Truncation with int(1.9):", int(1.9))
assert isinstance(True, int) and int(1.9) == 1

Boolean passes isinstance(True, int): True
Truncation with int(1.9): 1


### Step 2 — Build the descriptor slowly

`__set_name__` configures each descriptor’s private key (`_x`, `_y`). `__set__` validates **before** mutating storage. Returning a value from `__set__` is unnecessary: assignment does not use it.

In [10]:
class P04Integer:
    def __set_name__(self, owner, name):
        self.private_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.private_name]
        except KeyError:
            raise AttributeError(f"{self.private_name[1:]} has not been assigned") from None

    def __set__(self, instance, value):
        if type(value) is not int:
            raise TypeError("an exact int is required")
        instance.__dict__[self.private_name] = value

class P04Point:
    x = P04Integer()
    y = P04Integer()

p = P04Point()
p.x, p.y = 7, -9
print(p.__dict__, (p.x, p.y))

{'_x': 7, '_y': -9} (7, -9)


### Step 3 — Prove both storage isolation and atomic rejection

“Atomic rejection” here means the setter does not modify the *single field* when validation fails. Capture the previous value, attempt several invalid assignments, and confirm it remains unchanged. Both descriptor instances share the same implementation class, but each has its own private name.

In [11]:
q = P04Point()
q.x, q.y = 100, 200
assert (p.x, p.y, q.x, q.y) == (7, -9, 100, 200)
for bad in (True, 1.9, "14", None):
    try:
        p.x = bad
    except TypeError:
        pass
    else:
        raise AssertionError(f"Accepted forbidden value {bad!r}")
assert p.x == 7 and q.x == 100

fresh = P04Point()
try:
    _ = fresh.x
except AttributeError:
    print("Unassigned coordinate correctly raises AttributeError")
else:
    raise AssertionError("Missing coordinate should fail")
print("Problem 4 passed")

Unassigned coordinate correctly raises AttributeError
Problem 4 passed


**Checkpoint / interpretation.** The correct object receives the value via `instance`; each field uses a distinct key found via `__set_name__`. A missing field raises `AttributeError`, which also cooperates with `hasattr` and `getattr(obj, name, default)`. This implementation intentionally requires an instance `__dict__`; slotted objects need a different approach.

---
## Problem 05 — Which value wins? Data-descriptor precedence

**Learning objective.** Predict attribute lookup with descriptors, instance dictionaries, and ordinary class attributes.

**Challenge.** Demonstrate that a `__set__` method changes the lookup rules—even when instance.__dict__ contains the public field name.

Our method is deliberate: make a prediction, run a small experiment, explain what happened, and only then build the final solution.

### Step 1 — Create two competing candidates

Python does not always read `obj.__dict__[name]` first. A descriptor defining `__get__` and `__set__` (or `__delete__`) is a **data descriptor** and takes precedence over the instance dictionary. A getter-only descriptor is non-data and may be shadowed by an instance value.

In [12]:
class P05Data:
    def __get__(self, instance, owner):
        return self if instance is None else "from data descriptor"

    def __set__(self, instance, value):
        instance.__dict__["_recorded"] = value

class P05NonData:
    def __get__(self, instance, owner):
        return self if instance is None else "from non-data descriptor"

class P05Demo:
    strict = P05Data()
    flexible = P05NonData()
    plain = "from class attribute"

obj = P05Demo()
obj.__dict__.update(strict="shadow strict", flexible="shadow flexible", plain="shadow plain")
print(obj.strict, obj.flexible, obj.plain, sep=" | ")

from data descriptor | shadow flexible | shadow plain


### Step 2 — Explain each result

`strict` still runs the data descriptor, despite the direct dictionary entry. `flexible` gets the instance's string instead of calling its descriptor. `plain` is an ordinary class attribute and is also shadowed. This is a behavioral consequence of defining `__set__`, not a minor implementation detail.

In [13]:
assert obj.strict == "from data descriptor"
assert obj.flexible == "shadow flexible"
assert obj.plain == "shadow plain"
obj.strict = 123
assert obj.__dict__["_recorded"] == 123
assert obj.__dict__["strict"] == "shadow strict"  # Assignment was intercepted.
assert obj.strict == "from data descriptor"
print("Precedence and assignment interception verified")

Precedence and assignment interception verified


### Step 3 — Distinguish instance assignment from class assignment

`obj.strict = ...` invokes the descriptor's `__set__`. But `P05Demo.strict = ...` writes to the **class attribute**, replacing the descriptor in that class. Never assume a class-level assignment runs a descriptor's setter. Here we demonstrate replacement only on a disposable class.

In [14]:
class P05Disposable:
    x = P05Data()

old_descriptor = P05Disposable.x
P05Disposable.x = "new class field"
assert isinstance(old_descriptor, P05Data)
assert P05Disposable.x == "new class field"
assert P05Disposable().x == "new class field"
print("Class assignment replaced the descriptor, it did not call __set__")

Class assignment replaced the descriptor, it did not call __set__


**Checkpoint / interpretation.** Usual instance lookup, simplified: data descriptor on the class/MRO → instance dictionary → non-data descriptor or class attribute → `__getattr__` fallback. Special cases such as custom `__getattribute__`, `super()`, and metaclasses have their own behavior. Do not call `__get__` manually for ordinary field access.

---
## Problem 06 — A non-data descriptor as a deliberate override point

**Learning objective.** Use non-data descriptor shadowing intentionally, then invalidate an instance-specific cache.

**Challenge.** Implement a cheap per-instance lazy value whose first read computes once and whose explicit reset recomputes.

Our method is deliberate: make a prediction, run a small experiment, explain what happened, and only then build the final solution.

### Step 1 — Decide whether shadowing is a bug or a feature

In Problem 3, shadowing ruined a countdown. For an expensive computed value, shadowing is useful: on first access, compute a value and store it under **the public name** in `instance.__dict__`. Later accesses will return that cached value without calling the non-data descriptor. This resembles a simple cached property.

In [15]:
class P06Lazy:
    def __init__(self, compute):
        self.compute = compute

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        result = self.compute(instance)
        instance.__dict__[self.name] = result
        return result

class P06Document:
    def __init__(self, text):
        self.text = text
        self.computations = 0

    @P06Lazy
    def word_count(self):
        self.computations += 1
        return len(self.text.split())

doc = P06Document("red green blue")
print(doc.word_count, doc.word_count, doc.computations)

3 3 1


### Step 2 — Run the stale-cache experiment

Changing `doc.text` does not automatically invalidate `doc.word_count`: the instance dictionary shadows the descriptor. To recompute, remove the cached public key. We do not need a descriptor `__delete__` because this *non-data* design permits ordinary `del doc.word_count` when cached.

In [16]:
assert doc.word_count == 3 and doc.computations == 1
doc.text = "one two three four five"
assert doc.word_count == 3  # Deliberately stale.
del doc.word_count
assert doc.word_count == 5 and doc.computations == 2

other_doc = P06Document("hello")
assert other_doc.word_count == 1
assert other_doc.computations == 1
assert doc.computations == 2
print("Explicit invalidation and isolation verified")

Explicit invalidation and isolation verified


### Design trade-off

This technique requires `instance.__dict__`; ordinary `__slots__`-only objects lack it. It is also **not read-only**: `doc.word_count = 999` stores a shadowing value. For strict invariants or controlled invalidation, use a data descriptor (Problem 12) instead. This is why identifying the required semantics must come before choosing a descriptor category.

---
## Problem 07 — A deleter that resets only one instance

**Learning objective.** Add `__delete__` to a data descriptor and make deletion semantics explicit.

**Challenge.** Construct a temperature setting with a per-instance default, validation, and reset-to-default via `del`.

Our method is deliberate: make a prediction, run a small experiment, explain what happened, and only then build the final solution.

### Step 1 — Specify the three operations

Read returns the stored temperature or a configured default. Assignment accepts finite real numbers but rejects booleans. Deletion removes the per-instance override so the default applies again. The descriptor stores the **default** (shared configuration), while each object stores its own **override**.

In [17]:
class P07Temperature:
    def __init__(self, default):
        self.default = float(default)

    def __set_name__(self, owner, name):
        self.key = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.key, self.default)

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("temperature requires a real number")
        converted = float(value)
        if not math.isfinite(converted):
            raise ValueError("temperature must be finite")
        instance.__dict__[self.key] = converted

    def __delete__(self, instance):
        instance.__dict__.pop(self.key, None)

class P07Oven:
    temperature = P07Temperature(180)

oven1, oven2 = P07Oven(), P07Oven()
print(oven1.temperature, oven2.temperature)

180.0 180.0


### Step 2 — Observe resetting a single instance

Only the override for the target oven should be removed. Deleting an unset temperature is intentionally idempotent in this API. This choice is not mandatory for all descriptors: a read-only field might reject deletion altogether.

In [18]:
oven1.temperature = 210
oven2.temperature = 160
assert (oven1.temperature, oven2.temperature) == (210.0, 160.0)
del oven1.temperature
assert (oven1.temperature, oven2.temperature) == (180.0, 160.0)
del oven1.temperature
assert oven1.temperature == 180.0
for invalid in (True, float("inf"), float("nan"), "200"):
    try:
        oven1.temperature = invalid
    except (TypeError, ValueError):
        pass
    else:
        raise AssertionError(f"Incorrectly accepted {invalid!r}")
assert oven1.temperature == 180.0
print("Reset, isolation, and validation tests passed")

Reset, isolation, and validation tests passed


**Checkpoint / interpretation.** Python calls `__delete__(descriptor, instance)` when `del instance.field` is executed, provided the descriptor defines it. Deletion is not class attribute removal: `del P07Oven.temperature` would remove the class attribute itself. Note that a descriptor with `__get__` and only `__delete__` still counts as a data descriptor.

---
## Problem 08 — Read-only does not mean getter-only

**Learning objective.** Enforce a read-only public API even if callers try to shadow it.

**Challenge.** Build a per-instance generated token exposed by a data descriptor and reject both assignment and deletion.

Our method is deliberate: make a prediction, run a small experiment, explain what happened, and only then build the final solution.

### Step 1 — Predict the loophole

A getter-only `token` descriptor would be non-data. A caller could assign `obj.token = "fake"`, which adds an entry to `obj.__dict__` and shadows the getter. To block normal assignment, implement `__set__` and raise `AttributeError`. Deletion should also be explicitly blocked.

In [19]:
class P08ReadOnlyToken:
    def __set_name__(self, owner, name):
        self.key = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.key]
        except KeyError:
            raise AttributeError("token not initialized") from None

    def __set__(self, instance, value):
        raise AttributeError("token is read-only")

    def __delete__(self, instance):
        raise AttributeError("token cannot be deleted")

class P08Session:
    token = P08ReadOnlyToken()

    def __init__(self, token):
        # Trusted initialization writes the private backing field.
        self.__dict__["_token"] = token

session = P08Session("session-001")
print(session.token)

session-001


### Step 2 — Verify the descriptor wins over a forged dictionary entry

Direct `__dict__` modification can bypass this kind of API; Python descriptors are not a security boundary. Nevertheless, because this is a data descriptor, even an entry with the *public* name cannot shadow normal reads.

In [20]:
assert session.token == "session-001"
session.__dict__["token"] = "forged public value"
assert session.token == "session-001"
for operation in (lambda: setattr(session, "token", "new"),
                  lambda: delattr(session, "token")):
    try:
        operation()
    except AttributeError:
        pass
    else:
        raise AssertionError("Read-only operation unexpectedly succeeded")
assert session.token == "session-001"
print("The normal public API is read-only")

The normal public API is read-only


**Checkpoint / interpretation.** The getter’s backing field is deliberately private by convention, not cryptographically protected. For a truly immutable object, broader architectural techniques are needed. The key descriptor insight is that an *explicit rejecting setter* makes the field a data descriptor, preventing normal instance-dictionary shadowing.

---
## Problem 09 — Descriptors for objects without `__dict__`

**Learning objective.** Use `WeakKeyDictionary` to hold separate values for weak-referenceable, hashable slotted instances.

**Challenge.** Make a writable descriptor work on a slotted Sensor and prove its storage does not keep abandoned sensors alive.

Our method is deliberate: make a prediction, run a small experiment, explain what happened, and only then build the final solution.

### Step 1 — Why our earlier storage strategy fails

A class with `__slots__` and no `__dict__` generally cannot store arbitrary `instance.__dict__["_reading"]` values. One option is to declare a matching private slot explicitly. Another is an external map keyed by instance. We will explore the second to learn its limitations.

In [21]:
class P09Slotted:
    __slots__ = ("reading",)

s = P09Slotted()
print("Has an instance dictionary?", hasattr(s, "__dict__"))
assert not hasattr(s, "__dict__")

Has an instance dictionary? False


### Step 2 — Build an external, weak-keyed store

`WeakKeyDictionary` has **one entry per owning instance**. Unlike a normal dictionary, a weak key does not keep its object alive. Keys must be weak-referenceable and hashable. A slotted class should include `__weakref__` if it needs weak references and has no inherited weak-reference support. Crucially, the *stored value* must not itself hold a strong reference back to the key.

In [22]:
class P09WeakNumber:
    def __init__(self):
        self._values = WeakKeyDictionary()

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return self._values[instance]
        except KeyError:
            raise AttributeError("reading not initialized") from None

    def __set__(self, instance, value):
        if type(value) is not int:
            raise TypeError("an exact integer is required")
        self._values[instance] = value

    def __delete__(self, instance):
        self._values.pop(instance, None)

class P09Sensor:
    __slots__ = ("__weakref__",)
    reading = P09WeakNumber()

s1, s2 = P09Sensor(), P09Sensor()
s1.reading, s2.reading = 11, 22
print(s1.reading, s2.reading, "stored entries:", len(P09Sensor.reading._values))

11 22 stored entries: 2


### Step 3 — Verify independent storage and garbage collection

If the map held ordinary strong keys, it would keep sensors alive merely because someone once assigned a reading. Record a weak reference to a disposable sensor, remove its last strong reference, then check that its map entry disappears. The real program must still avoid reference cycles through the mapped values.

In [23]:
assert (s1.reading, s2.reading) == (11, 22)
start_size = len(P09Sensor.reading._values)
temporary = P09Sensor()
temporary.reading = 33
temporary_ref = ref(temporary)
assert len(P09Sensor.reading._values) == start_size + 1
del temporary
gc.collect()
assert temporary_ref() is None
assert len(P09Sensor.reading._values) == start_size
print("No cross-instance leakage and no retained temporary sensor")

No cross-instance leakage and no retained temporary sensor


### Step 4 — Try an incompatible owner, safely

A slotted object with no weak-reference support cannot become a weak key. Rather than hiding that restriction, test that an assignment fails. Also note that instances with `__hash__ = None` are incompatible unless you choose another storage scheme.

In [24]:
class P09NoWeakref:
    __slots__ = ()
    reading = P09WeakNumber()

try:
    P09NoWeakref().reading = 1
except TypeError as exc:
    print("Expected WeakKeyDictionary limitation:", exc)
else:
    raise AssertionError("A non-weakrefable key should fail")

Expected WeakKeyDictionary limitation: cannot create weak reference to 'P09NoWeakref' object


**Checkpoint / interpretation.** Weak-key storage is appropriate for a particular owner contract, not a universal default. It introduces lookup and weak-reference constraints, and weak values must not strongly retain their corresponding keys. For common objects with `__dict__`, a private instance key is usually simpler.

---
## Problem 10 — Inheritance, `__set_name__`, and accidental descriptor aliasing

**Learning objective.** Understand that class creation binds descriptor names, but inheriting a descriptor does not create a new descriptor.

**Challenge.** Explore one inherited field, a deliberate override, and the bug caused by installing one descriptor object under two names.

Our method is deliberate: make a prediction, run a small experiment, explain what happened, and only then build the final solution.

### Step 1 — Inheritance does not clone a field

`__set_name__` runs when an instance of a descriptor is placed directly in a class body at class creation. A subclass that merely inherits it does not automatically make a copy. Check descriptor identity through class access.

In [25]:
class P10NamedField:
    def __set_name__(self, owner, name):
        self.owner = owner
        self.name = name
        self.key = "_" + name

    def __get__(self, instance, owner):
        return self if instance is None else instance.__dict__.get(self.key, None)

    def __set__(self, instance, value):
        instance.__dict__[self.key] = value

class P10Parent:
    label = P10NamedField()

class P10Child(P10Parent):
    pass

assert P10Parent.label is P10Child.label
assert P10Child.label.owner is P10Parent
child = P10Child()
child.label = "child value"
print(child.label, P10Child.label.owner.__name__)

child value P10Parent


### Step 2 — What happens when the subclass overrides the field?

Defining a *new descriptor object* in the subclass gives the subclass its own binding. The parent still points to its original object. The new descriptor can impose different rules without silently changing the parent’s descriptor configuration.

In [26]:
class P10Override(P10Parent):
    label = P10NamedField()

assert P10Override.label is not P10Parent.label
assert P10Override.label.owner is P10Override
assert P10Parent.label.owner is P10Parent
print("New descriptor is bound to the overriding class")

New descriptor is bound to the overriding class


### Step 3 — Create an aliasing mistake on purpose

If the **same descriptor object** is assigned to two names in one class, Python calls `__set_name__` twice on that one object. A descriptor storing a single `self.name` cannot simultaneously remember two different names. Predict the instance dictionary below before checking it.

In [27]:
shared = P10NamedField()
class P10Aliased:
    first = shared
    second = shared

aliased = P10Aliased()
aliased.first = "A"
aliased.second = "B"
print("Broken alias result:", aliased.first, aliased.second, aliased.__dict__)
assert aliased.first == aliased.second == "B"
assert aliased.__dict__ == {"_second": "B"}

Broken alias result: B B {'_second': 'B'}


### Step 4 — Repair aliasing by specifying separate field objects

If `first` and `second` must be separate fields, create **two descriptor instances**, as the source lesson does for `x` and `y`. If you genuinely want aliases to the same field, implement explicit alias semantics instead of relying on accidental double-binding.

In [28]:
class P10Independent:
    first = P10NamedField()
    second = P10NamedField()

independent = P10Independent()
independent.first, independent.second = "A", "B"
assert (independent.first, independent.second) == ("A", "B")
assert independent.__dict__ == {"_first": "A", "_second": "B"}
print("Two fields are independent again")

Two fields are independent again


**Checkpoint / interpretation.** A single descriptor shared among instances of **one field** is expected. The problematic sharing here is reusing that same name-bound descriptor object for **different fields**. Inheritance itself does not rerun `__set_name__` for an inherited descriptor. These distinctions become essential in reusable validation libraries.

---
## Problem 11 — Installing a descriptor *after* a class exists

**Learning objective.** Recognize that dynamic `setattr` does not automatically call `__set_name__`.

**Challenge.** Build a dynamic field, diagnose the missing name binding, and explicitly finish initialization.

Our method is deliberate: make a prediction, run a small experiment, explain what happened, and only then build the final solution.

### Step 1 — Understand the creation-time hook

`__set_name__` is a class-creation callback. A descriptor added later using `setattr(SomeClass, name, descriptor)` does **not** receive it automatically. This is easy to miss when dynamically constructing schemas, ORMs, or test fixtures.

In [29]:
class P11DynamicField:
    def __set_name__(self, owner, name):
        self.key = "_" + name
        self.bound_owner = owner

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.key]

    def __set__(self, instance, value):
        instance.__dict__[self.key] = value

class P11Model:
    pass

field = P11DynamicField()
setattr(P11Model, "status", field)
print("Bound automatically?", hasattr(field, "key"))
assert not hasattr(field, "key")

Bound automatically? False


### Step 2 — Reproduce and fix the failure

Normal assignment reaches `__set__`, but it cannot work yet because `self.key` does not exist. Invoke `field.__set_name__(P11Model, "status")` exactly once after installation. Dynamic schema builders should centralize these two operations to avoid forgetting this step.

In [30]:
instance = P11Model()
try:
    instance.status = "pending"
except AttributeError:
    print("Expected: dynamically assigned field was unbound")
else:
    raise AssertionError("An unbound field should have failed")

field.__set_name__(P11Model, "status")
instance.status = "ready"
assert instance.status == "ready"
assert P11Model.status is field
assert field.bound_owner is P11Model
print("Dynamic field successfully initialized")

Expected: dynamically assigned field was unbound
Dynamic field successfully initialized


### Suggested reusable pattern

When a framework creates descriptor fields dynamically, make an `install_field(cls, name, descriptor)` helper that assigns the descriptor and calls `__set_name__`. Only do this once per intended binding; as Problem 10 showed, rebinding a name-aware descriptor can overwrite its existing configuration.

In [31]:
def p11_install_field(owner, name, descriptor):
    if name in owner.__dict__:
        raise ValueError(f"{name!r} is already defined directly on this class")
    setattr(owner, name, descriptor)
    descriptor.__set_name__(owner, name)

class P11OtherModel:
    pass

p11_install_field(P11OtherModel, "priority", P11DynamicField())
other_model = P11OtherModel()
other_model.priority = "high"
assert other_model.priority == "high"
print("Installation helper verified")

Installation helper verified


**Checkpoint / interpretation.** The helper is intentionally simple: it only checks attributes directly present on the class. A production framework should also specify how inherited-name collisions, failed binding, and descriptor reuse should be handled. A dynamic class `type("Name", (), {"field": descriptor})` is another approach: its class creation does call `__set_name__`.

---
## Problem 12 — Invalidate a derived value without stale caches

**Learning objective.** Coordinate separate descriptor fields through instance-specific revision numbers.

**Challenge.** Cache a vector’s squared length and invalidate the cache automatically when either coordinate is assigned.

Our method is deliberate: make a prediction, run a small experiment, explain what happened, and only then build the final solution.

### Step 1 — State the dependency problem

A cached `x*x + y*y` becomes stale when `x` or `y` changes. Rather than deleting a public cached attribute manually (Problem 6), use an instance revision counter. Both setters bump that counter; the derived getter checks whether its cached revision matches the current revision.

In [32]:
class P12TrackedNumber:
    def __set_name__(self, owner, name):
        self.key = "_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.key]
        except KeyError:
            raise AttributeError("number has not been initialized") from None

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("a number is required")
        number = float(value)
        if not math.isfinite(number):
            raise ValueError("a finite number is required")
        instance.__dict__[self.key] = number
        instance.__dict__["_revision"] = instance.__dict__.get("_revision", 0) + 1

class P12SquaredLength:
    def __set_name__(self, owner, name):
        self.cache_key = "_cache_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        revision = instance.__dict__.get("_revision", 0)
        cache = instance.__dict__.get(self.cache_key, _MISSING)
        if cache is not _MISSING and cache[0] == revision:
            return cache[1]
        value = instance.x**2 + instance.y**2
        instance.__dict__[self.cache_key] = (revision, value)
        instance.__dict__["_computations"] += 1
        return value

### Step 2 — Finish the owner class

Keep `_computations` on the vector, not the descriptor, so two vectors have separate diagnostics. The setters must be used consistently: direct writes to `obj.__dict__["_x"]` bypass revision tracking and can produce stale results.

In [33]:
class P12Vector:
    x = P12TrackedNumber()
    y = P12TrackedNumber()
    squared_length = P12SquaredLength()

    def __init__(self, x, y):
        self._computations = 0
        self.x = x
        self.y = y

v1 = P12Vector(3, 4)
v2 = P12Vector(5, 12)
print(v1.squared_length, v1.squared_length, v1._computations)
print(v2.squared_length, v2._computations)

25.0 25.0 1
169.0 1


### Step 3 — Test cache hits, invalidation, and isolation

Reading twice at the same revision should compute once. Assigning a coordinate creates a new revision even if the new value happens to equal the old value; this simple contract prefers correctness over optimizing redundant writes.

In [34]:
assert (v1.squared_length, v2.squared_length) == (25.0, 169.0)
assert (v1._computations, v2._computations) == (1, 1)
v1.x = 6
assert v1.squared_length == 52.0
assert v1.squared_length == 52.0
assert (v1._computations, v2._computations) == (2, 1)
try:
    v1.y = float("nan")
except ValueError:
    pass
else:
    raise AssertionError("Invalid coordinate accepted")
assert v1.squared_length == 52.0 and v1._computations == 2
print("Automatic invalidation and instance isolation passed")

Automatic invalidation and instance isolation passed


### Deeper reasoning

The cache is not globally shared: each vector has its own `_revision`, `_cache_squared_length`, and `_computations`. This solution assumes single-threaded mutation and a closed list of dependencies. If another field begins affecting the result, its setter must also increment the revision. If multiple threads concurrently access it, coordinate reads and revision checks need synchronization; see Problem 16.

---
## Problem 13 — Make multi-field updates atomic

**Learning objective.** Avoid partially updated objects when two descriptor assignments must jointly satisfy an invariant.

**Challenge.** Implement a rectangle whose positive width and height must yield area at most 100.

Our method is deliberate: make a prediction, run a small experiment, explain what happened, and only then build the final solution.

### Step 1 — Expose the partial-update trap

A setter can validate an individual field, but a two-field update via `self.width = w; self.height = h` may change width before discovering that height is invalid. We first demonstrate the failure with a small disposable class. The `try` block catches the *expected* error; the partially modified state is the real bug.

In [35]:
class P13Positive:
    def __set_name__(self, owner, name):
        self.key = "_" + name

    def __get__(self, instance, owner):
        return self if instance is None else instance.__dict__[self.key]

    def __set__(self, instance, value):
        if type(value) not in (int, float) or not math.isfinite(value) or value <= 0:
            raise ValueError("positive finite number required")
        instance.__dict__[self.key] = float(value)

class P13NaiveRectangle:
    width = P13Positive()
    height = P13Positive()

    def __init__(self, width, height):
        self.width, self.height = width, height

    def resize(self, width, height):
        self.width = width
        self.height = height

naive = P13NaiveRectangle(5, 10)
try:
    naive.resize(7, -1)
except ValueError:
    pass
assert (naive.width, naive.height) == (7.0, 10.0)  # Broken partial state!
print("Naive resize leaves width changed:", naive.width)

Naive resize leaves width changed: 7.0


### Step 2 — Validate *all* candidates before committing

Our corrected implementation uses a positive-field descriptor that additionally checks the total area against the other current dimension for ordinary single-field assignment. Its `resize` method performs all checks up front, including area, and only then commits both private values. Do not call the two public setters sequentially in `resize`: an intermediate size could violate the invariant even when the final size is valid.

In [36]:
class P13Dimension:
    def __set_name__(self, owner, name):
        self.name = name
        self.key = "_" + name
        self.other_key = "_height" if name == "width" else "_width"

    @staticmethod
    def check(value):
        if type(value) not in (int, float) or not math.isfinite(value) or value <= 0:
            raise ValueError("dimensions must be positive finite numbers")
        return float(value)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.key]

    def __set__(self, instance, value):
        number = self.check(value)
        other = instance.__dict__.get(self.other_key)
        if other is not None and number * other > 100:
            raise ValueError("maximum area is 100")
        instance.__dict__[self.key] = number

class P13Rectangle:
    width = P13Dimension()
    height = P13Dimension()

    def __init__(self, width, height):
        self.resize(width, height)

    def resize(self, width, height):
        proposed_width = type(self).width.check(width)
        proposed_height = type(self).height.check(height)
        if proposed_width * proposed_height > 100:
            raise ValueError("maximum area is 100")
        # No expected validation failure remains after this point.
        self.__dict__.update(_width=proposed_width, _height=proposed_height)

    @property
    def area(self):
        return self.width * self.height

rectangle = P13Rectangle(5, 10)
print(rectangle.width, rectangle.height, rectangle.area)

5.0 10.0 50.0


### Step 3 — Test rollback semantics and valid transitions

“Atomic” here means validation failures cannot leave a partial logical update in ordinary single-threaded use. It does not promise database transactions or concurrency isolation. We will test that both values remain unchanged after either invalid input and an excessive area.

In [37]:
original = (rectangle.width, rectangle.height)
for proposed in ((20, 20), (7, -1), ("7", 1), (float("nan"), 5)):
    try:
        rectangle.resize(*proposed)
    except (ValueError, TypeError):
        pass
    else:
        raise AssertionError(f"Invalid dimensions accepted: {proposed!r}")
    assert (rectangle.width, rectangle.height) == original

try:
    rectangle.width = 30
except ValueError:
    pass
else:
    raise AssertionError("Single-field update broke area invariant")
assert (rectangle.width, rectangle.height) == original
rectangle.resize(20, 5)
assert rectangle.area == 100
print("All validation-failure paths preserve the previous rectangle")

All validation-failure paths preserve the previous rectangle


**Checkpoint / interpretation.** Descriptors are field-level hooks; some invariants belong to the **whole object**. Use a coordinated owner method for multi-field transitions. The internal `__dict__.update(...)` is a controlled commit after comprehensive validation—not permission for unrelated callers to bypass the public API.

---
## Problem 14 — Money fields without silent floating-point surprises

**Learning objective.** Design a descriptor with documented coercion and explicit decimal rounding.

**Challenge.** Create an amount field that accepts `str`, `int`, and `Decimal`, rejects binary floats/bools, and stores exact two-decimal values.

Our method is deliberate: make a prediction, run a small experiment, explain what happened, and only then build the final solution.

### Step 1 — Why validation policy is part of the API

Problem 4 deliberately rejected all non-int values. Other domains need *controlled* conversion. For money, converting a binary float to `Decimal` can preserve an approximation that the caller did not intend. We will accept strings, exact integers, and existing `Decimal` values, then round to cents using `ROUND_HALF_EVEN`. This is a teaching example, not an accounting specification for every jurisdiction.

In [38]:
from decimal import Decimal, InvalidOperation, ROUND_HALF_EVEN

print("Decimal from text:", Decimal("0.1"))
print("Decimal from float:", Decimal(0.1))
assert Decimal("0.1") != Decimal(0.1)

Decimal from text: 0.1
Decimal from float: 0.1000000000000000055511151231257827021181583404541015625


### Step 2 — Convert, validate, round, and then store

The order matters: reject unsafe types first; parse the value; reject NaN and infinity; quantize to a fixed precision; enforce nonnegativity; finally assign to the *owner instance*. An invalid request must not destroy the previous amount.

In [39]:
class P14Money:
    CENT = Decimal("0.01")

    def __set_name__(self, owner, name):
        self.key = "_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.key]
        except KeyError:
            raise AttributeError("money field not assigned") from None

    def __set__(self, instance, value):
        if type(value) not in (str, int, Decimal):
            raise TypeError("Use a string, exact integer, or Decimal; never float/bool")
        try:
            amount = Decimal(value)
            if not amount.is_finite():
                raise ValueError("amount must be finite")
            amount = amount.quantize(self.CENT, rounding=ROUND_HALF_EVEN)
        except InvalidOperation as exc:
            raise ValueError("invalid monetary amount or precision") from exc
        if amount < 0:
            raise ValueError("amount must be nonnegative")
        instance.__dict__[self.key] = amount

class P14Invoice:
    amount = P14Money()

invoice_a, invoice_b = P14Invoice(), P14Invoice()
invoice_a.amount = "19.995"
invoice_b.amount = 7
print(invoice_a.amount, invoice_b.amount)

20.00 7.00


### Step 3 — Test boundary and isolation behavior

The sample uses the *current decimal context* to quantize. Very large numbers may exceed the context’s precision and raise `InvalidOperation`, which we convert to `ValueError`. A production money type should explicitly define acceptable maximum values, currency precision, and rounding rules.

In [40]:
assert invoice_a.amount == Decimal("20.00")
assert invoice_b.amount == Decimal("7.00")
invoice_a.amount = Decimal("1.005")
assert invoice_a.amount == Decimal("1.00")  # HALF_EVEN at the tie.
for bad in (1.1, True, "NaN", "Infinity", "-1", "not money"):
    try:
        invoice_a.amount = bad
    except (TypeError, ValueError):
        pass
    else:
        raise AssertionError(f"Accepted invalid monetary input {bad!r}")
assert invoice_a.amount == Decimal("1.00")
assert invoice_b.amount == Decimal("7.00")
print("Money precision, rejection, and isolation tests passed")

Money precision, rejection, and isolation tests passed


**Checkpoint / interpretation.** A descriptor centralizes policy across many owning objects while keeping values independent. Rejecting floats is a chosen API constraint, not something Python enforces automatically. Do not use `int(value)` simply because conversion is convenient; decide whether precision loss is permitted.

---
## Problem 15 — Build a minimal descriptor-backed record schema

**Learning objective.** Collect inherited descriptors without triggering them, validate fields, and serialize records.

**Challenge.** Make a small model base class with strict fields and correct behavior when a subclass overrides a field with a plain attribute.

Our method is deliberate: make a prediction, run a small experiment, explain what happened, and only then build the final solution.

### Step 1 — Why `getattr(cls, name)` is not enough for schema discovery

Class access often returns a descriptor, but the protocol permits returning **anything**. For reliable discovery, inspect each class’s raw `__dict__`, not computed values. Walk the MRO from oldest ancestor to newest descendant so subclass overrides win. If a subclass changes a former field into a plain attribute, the field should disappear from the schema.

In [41]:
class P15Field:
    def __init__(self, expected_type):
        self.expected_type = expected_type

    def __set_name__(self, owner, name):
        self.name = name
        self.key = "_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.key]
        except KeyError:
            raise AttributeError(f"required field {self.name!r} is unset") from None

    def __set__(self, instance, value):
        if type(value) is not self.expected_type:
            raise TypeError(f"{self.name} requires {self.expected_type.__name__}")
        instance.__dict__[self.key] = value

### Step 2 — Collect fields in a carefully chosen order

The first base classes appear last in `reversed(__mro__)`. Processing newer class dictionaries afterward lets each subclass replace or remove earlier field entries. This simple implementation supports single or multiple inheritance according to Python’s linearized MRO; it does not attempt to be a full ORM.

In [42]:
class P15Model:
    @classmethod
    def fields(cls):
        found = {}
        for base in reversed(cls.__mro__):
            for name, value in base.__dict__.items():
                if isinstance(value, P15Field):
                    found[name] = value
                elif name in found:
                    del found[name]  # Subclass shadows inherited field.
        return found

    def __init__(self, **values):
        fields = type(self).fields()
        missing = fields.keys() - values.keys()
        extra = values.keys() - fields.keys()
        if missing or extra:
            raise TypeError(f"missing={sorted(missing)}, unexpected={sorted(extra)}")
        # Validate everything before storing anything on this new object.
        for name, descriptor in fields.items():
            if type(values[name]) is not descriptor.expected_type:
                raise TypeError(f"{name} requires {descriptor.expected_type.__name__}")
        for name, value in values.items():
            setattr(self, name, value)

    def as_dict(self):
        return {name: getattr(self, name) for name in type(self).fields()}

class P15Person(P15Model):
    name = P15Field(str)
    age = P15Field(int)

class P15Employee(P15Person):
    department = P15Field(str)

person = P15Person(name="Ada", age=37)
employee = P15Employee(name="Lin", age=28, department="Research")
print(person.as_dict(), employee.as_dict())

{'name': 'Ada', 'age': 37} {'name': 'Lin', 'age': 28, 'department': 'Research'}


### Step 3 — Test both isolation and subclass field replacement

Assigning a new descriptor in a subclass overrides the inherited descriptor; assigning a plain class attribute replaces that field entirely **for the subclass schema**. Each concrete object still has its own values. This avoids a frequent inheritance bug: collecting `vars(Base)` without accounting for shadowing.

In [43]:
class P15AgeFreePerson(P15Person):
    age = "not a descriptor on this subclass"

assert list(P15Person.fields()) == ["name", "age"]
assert list(P15Employee.fields()) == ["name", "age", "department"]
assert list(P15AgeFreePerson.fields()) == ["name"]
assert P15Person.fields()["age"] is P15Person.__dict__["age"]
assert P15AgeFreePerson(name="Sky").as_dict() == {"name": "Sky"}
employee.name = "Maya"
assert person.name == "Ada" and employee.name == "Maya"
for arguments in ({"name": "A"}, {"name": "A", "age": True},
                  {"name": "A", "age": 1, "unknown": 10}):
    try:
        P15Person(**arguments)
    except TypeError:
        pass
    else:
        raise AssertionError(f"Invalid schema input accepted: {arguments!r}")
print("Inherited discovery, overriding, validation, and isolation passed")

Inherited discovery, overriding, validation, and isolation passed


### Boundary of this example

This schema does not handle aliases for one descriptor instance, mutable defaults, converters, or mutation of nested values. It exposes a deliberate, small contract so we can **reason** about descriptor lookup rather than build a production ORM accidentally. The key lesson is that class `__dict__` provides *raw descriptor definitions*, whereas ordinary `getattr(cls, name)` invokes class-access getter behavior.

---
## Problem 16 — Two threads, one descriptor: fix the state race

**Learning objective.** Distinguish shared descriptor state from safe per-instance state, and protect read-modify-write sequences.

**Challenge.** Show a deterministic lost update, then implement a per-owner counter with an instance lock.

Our method is deliberate: make a prediction, run a small experiment, explain what happened, and only then build the final solution.

### Step 1 — Make a race reproducible instead of hoping it appears

If two threads load the same shared counter value and then both write `old + 1`, one increment is lost. A `Barrier` forces both threads to read before either writes. The next code deliberately contains a bad **descriptor-global** counter to make the problem visible without relying on timing or `sleep`.

In [44]:
class P16RacyCounter:
    def __init__(self, barrier):
        self.value = 0
        self.barrier = barrier

    def __get__(self, instance, owner):
        if instance is None:
            return self
        old = self.value
        self.barrier.wait(timeout=5)
        self.value = old + 1
        return self.value

barrier = Barrier(2)
class P16Broken:
    next_number = P16RacyCounter(barrier)

broken = P16Broken()
results = []
threads = [Thread(target=lambda: results.append(broken.next_number)) for _ in range(2)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join(timeout=5)
assert all(not thread.is_alive() for thread in threads)
assert sorted(results) == [1, 1]
assert P16Broken.next_number.value == 1
print("Two calls, but only one increment persisted:", results)

Two calls, but only one increment persisted: [1, 1]


### Step 2 — Define correct ownership and synchronization

Put `_count` and `_counter_lock` **on the owner instance**. The descriptor should only know how to access those fields. A lock protects the entire read-modify-write operation, not just the assignment line. Create the lock before launching threads; initialization is an explicit obligation of this small API.

In [45]:
class P16SafeCounter:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        with instance._counter_lock:
            instance._count += 1
            return instance._count

class P16Worker:
    next_number = P16SafeCounter()

    def __init__(self):
        self._count = 0
        self._counter_lock = Lock()

safe = P16Worker()
iterations_per_thread = 200
threads = [Thread(target=lambda: [safe.next_number for _ in range(iterations_per_thread)])
           for _ in range(4)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join(timeout=5)
assert all(not thread.is_alive() for thread in threads)
assert safe._count == 4 * iterations_per_thread
other = P16Worker()
assert other.next_number == 1
assert safe._count == 800
print("Protected calls:", safe._count, "independent owner:", other._count)

Protected calls: 800 independent owner: 1


### Discussion — What exactly did the lock solve?

A per-instance lock protects one instance’s counter under concurrent access. It also keeps *unrelated* owners from needlessly sharing a single descriptor-global lock. It does not automatically make every unrelated attribute of the owner thread-safe. A concurrent, multi-field cache like Problem 12 would need a broader synchronization design around its dependencies and cached result.

---
## Problem 17 — Compare `property` with a custom descriptor

**Learning objective.** See how built-in properties follow descriptor class/instance access rules.

**Challenge.** Trace a custom getter and a property, then inspect descriptors without triggering getters.

Our method is deliberate: make a prediction, run a small experiment, explain what happened, and only then build the final solution.

### Step 1 — Recreate the source lesson’s class-access observation

A `property` is itself a descriptor. `SomeClass.property_name` typically returns the property object, while `some_instance.property_name` calls the wrapped getter. Unlike our previous read-only mistake, even a property with no setter is a **data descriptor**: its assignment attempt raises rather than permitting shadowing.

In [46]:
class P17Example:
    def __init__(self):
        self._value = 5
        self.reads = 0

    @property
    def value(self):
        self.reads += 1
        return self._value

obj17 = P17Example()
assert isinstance(P17Example.value, property)
assert obj17.value == 5 and obj17.reads == 1
try:
    obj17.value = 999
except AttributeError:
    print("Getter-only property rejects assignment")
else:
    raise AssertionError("Property unexpectedly accepted assignment")

Getter-only property rejects assignment


### Step 2 — Inspect a descriptor *without executing it*

Normal `getattr(obj, "value")` runs the getter and increments `reads`. `inspect.getattr_static(obj, "value")` is useful in tooling because it retrieves the raw descriptor without invoking normal dynamic lookup. It has limitations when custom attribute access or unusual dictionaries are involved; here it illustrates a clean contrast.

In [47]:
import inspect

before = obj17.reads
raw = inspect.getattr_static(obj17, "value")
assert raw is P17Example.__dict__["value"]
assert obj17.reads == before
assert getattr(obj17, "value") == 5
assert obj17.reads == before + 1
print("Static inspection did not run the getter; getattr did")

Static inspection did not run the getter; getattr did


### Step 3 — When is a custom descriptor worthwhile?

Prefer `@property` for behavior tied to **one** class and field. Choose a custom descriptor when you need a reusable rule applied to **many** fields/classes, such as integer validation, audit logging, or storage policy. In both cases, instance state still belongs to the target instance unless the intended semantics explicitly require shared state.

---
## Problem 18 — Final application: an audited order record

**Learning objective.** Combine reusable validation, per-instance storage, read-only derived data, and change history.

**Challenge.** Build an order with exact integer quantity, Decimal-backed unit price, and a per-instance audit trail that records only successful changes.

Our method is deliberate: make a prediction, run a small experiment, explain what happened, and only then build the final solution.

### Step 1 — Establish the design contract

- `quantity`: exact `int` greater than zero, rejecting `bool`.
- `unit_price`: nonnegative `Decimal` with cents, rejecting `float`.
- `total`: a read-only derived property, never independently stored.
- `changes`: one list per order. Rejected assignments must neither change fields nor append events.

The descriptor objects themselves retain only validators and field-name metadata. Nothing specific to one order is stored on a shared descriptor.

In [48]:
class P18AuditedField:
    def __init__(self, validator):
        self.validator = validator

    def __set_name__(self, owner, name):
        self.name = name
        self.key = "_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.key]
        except KeyError:
            raise AttributeError(f"{self.name} not initialized") from None

    def __set__(self, instance, value):
        validated = self.validator(value)  # May fail: no state has changed yet.
        old = instance.__dict__.get(self.key, _MISSING)
        instance.__dict__[self.key] = validated
        if old is _MISSING or old != validated:
            instance._changes.append((self.name,
                                      None if old is _MISSING else old,
                                      validated))


def p18_quantity(value):
    if type(value) is not int or value <= 0:
        raise ValueError("quantity must be an exact positive integer")
    return value


def p18_price(value):
    if type(value) not in (str, int, Decimal):
        raise TypeError("price must use string, exact int, or Decimal")
    try:
        result = Decimal(value)
        if not result.is_finite():
            raise ValueError("price must be finite")
        result = result.quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN)
    except InvalidOperation as exc:
        raise ValueError("invalid price") from exc
    if result < 0:
        raise ValueError("price must be nonnegative")
    return result

### Step 2 — Add the owner and initialize its audit list *first*

A common ordering error is assigning the descriptor-backed fields before creating `_changes`. Since their setters write audit events, initialization must first establish the audit destination. `total` is computed from the canonical validated values, so it cannot become stale.

In [49]:
class P18Order:
    quantity = P18AuditedField(p18_quantity)
    unit_price = P18AuditedField(p18_price)

    def __init__(self, quantity, unit_price):
        self._changes = []
        self.quantity = quantity
        self.unit_price = unit_price

    @property
    def total(self):
        return self.unit_price * self.quantity

    @property
    def changes(self):
        return tuple(self._changes)  # Read-only snapshot via the public API.

order1 = P18Order(2, "12.50")
order2 = P18Order(1, "3.00")
print("Order totals:", order1.total, order2.total)
print("First order events:", order1.changes)

Order totals: 25.00 3.00
First order events: (('quantity', None, 2), ('unit_price', None, Decimal('12.50')))


### Step 3 — Test the entire observable contract

One successful update should affect only one order and add one event. An invalid assignment must fail before storage and logging. Assigning the same canonical value should not produce a duplicate event. Finally, callers must not assign `total` because it is a getter-only `property`.

In [50]:
assert order1.total == Decimal("25.00")
assert order2.total == Decimal("3.00")
assert len(order1.changes) == 2
order1.quantity = 4
assert order1.total == Decimal("50.00")
assert len(order1.changes) == 3
order1.unit_price = "12.500"  # Canonical value unchanged.
assert len(order1.changes) == 3

old_state = (order1.quantity, order1.unit_price, order1.changes)
for field, invalid in (("quantity", True), ("quantity", 0),
                       ("unit_price", 2.5), ("unit_price", "NaN")):
    try:
        setattr(order1, field, invalid)
    except (TypeError, ValueError):
        pass
    else:
        raise AssertionError(f"Rejected input accepted: {field}={invalid!r}")
    assert (order1.quantity, order1.unit_price, order1.changes) == old_state
assert order2.total == Decimal("3.00") and len(order2.changes) == 2
try:
    order1.total = Decimal("1.00")
except AttributeError:
    pass
else:
    raise AssertionError("Derived total should be read-only")
assert P18Order.quantity is P18Order.__dict__["quantity"]
print("Integrated solution: all contract checks passed")

Integrated solution: all contract checks passed


### Why the integrated solution works

The two field descriptor objects are class attributes and can be shared safely; each order’s `_quantity`, `_unit_price`, and `_changes` are instance attributes. Validation finishes before any mutation, preserving state on failure. Derived data is computed instead of copied into a second storage location. The public `changes` property returns a tuple snapshot, although callers who intentionally access `_changes` can still bypass the public convention. This is a reusable descriptor **design pattern**, not an access-control security mechanism.

---
## Bonus: prediction lab (explanations appear after the executable checks)

Without running the next cell, predict each answer:

1. A getter called as `Sub.field`: what is `instance`?
2. Do two instances of one owner class have two descriptor objects?
3. A setter writes `self.value`: which object owns that value?
4. What does assigning a descriptor-backed field on the *class* normally do?
5. Can a data descriptor be shadowed by `instance.__dict__["field"]` during ordinary access?
6. What special method makes a class-created descriptor aware of its assigned name?
7. Does late `setattr(SomeClass, "field", descriptor)` automatically call that method?
8. What can prevent `WeakKeyDictionary` storage on a slotted object?
9. Why can two individually valid assignments produce an invalid intermediate object?
10. Can a getter-only property be shadowed by normal assignment to its instance?

In [51]:
bonus_answers = {
    1: "None",
    2: "No; the field's descriptor object is shared.",
    3: "The shared descriptor: a common source of cross-instance bugs.",
    4: "It replaces the class attribute; it does not call the instance setter.",
    5: "No, not in ordinary instance lookup.",
    6: "__set_name__",
    7: "No; call __set_name__ explicitly after dynamic installation.",
    8: "No weak-reference support or unhashable owner instances.",
    9: "The first write can commit before validation of the second fails.",
    10: "No; property is a data descriptor and rejects assignment without a setter.",
}
for number, answer in bonus_answers.items():
    print(f"{number:2d}. {answer}")
assert len(bonus_answers) == 10

 1. None
 2. No; the field's descriptor object is shared.
 3. The shared descriptor: a common source of cross-instance bugs.
 4. It replaces the class attribute; it does not call the instance setter.
 5. No, not in ordinary instance lookup.
 6. __set_name__
 7. No; call __set_name__ explicitly after dynamic installation.
 8. No weak-reference support or unhashable owner instances.
 9. The first write can commit before validation of the second fails.
10. No; property is a data descriptor and rejects assignment without a setter.


---
## Debugging checklist: how to reason about a new descriptor

**Start with protocol dispatch.** Is the attribute being accessed via a class or an instance? What is the owner class? Does the getter return `self` when `instance is None`?

**Next, inspect state ownership.** Is a mutable field stored in `descriptor.__dict__` (shared), `owner_instance.__dict__` (separate), a declared slot, or a suitable weak-key map? Shared configuration is fine; accidental shared *instance values* are not.

**Then identify lookup precedence.** Does the descriptor define `__set__` or `__delete__`, making it a data descriptor? Could an instance dictionary shadow a non-data descriptor? Would class assignment replace the descriptor?

**Finally, verify the contract.** What happens on unset reads, invalid values, deletion, inheritance, rebinding, slotted owners, caching dependencies, and concurrent operations? Check that failed updates do not corrupt old state and that independently created objects remain independent.

## Notebook completion check

Every example above is deliberately split across explanatory and executable cells. Running the full notebook should reach this cell without unexpected exceptions. This marker confirms only that the preceding cells executed; the assertions inside each worked solution provide the actual checks.

In [52]:
print("ALL 18 GUIDED PROBLEMS COMPLETED; 10 BONUS PREDICTIONS AVAILABLE.")

ALL 18 GUIDED PROBLEMS COMPLETED; 10 BONUS PREDICTIONS AVAILABLE.
